In [4]:
import pandas as pd

In [5]:
stk_data=pd.read_csv("TATACOFEE_1321.csv")

In [6]:
stk_data

,Date,Price,Open,High,Low,Vol.,Change %
0,15-01-2013,"1,586.95","1,605.45","1,605.45","1,582.20",57.64K,-0.42%
1,16-01-2013,"1,572.70","1,588.05","1,591.15","1,570.00",62.35K,-0.90%
2,17-01-2013,"1,599.90","1,577.15","1,645.80","1,570.00",225.98K,1.73%
3,18-01-2013,"1,587.00","1,609.80","1,614.70","1,578.55",71.61K,-0.81%
4,21-01-2013,"1,596.90","1,594.75","1,621.00","1,587.95",64.27K,0.62%
...,...,...,...,...,...,...,...
2210,27-12-2021,218.35,200.00,222.00,196.00,5.89M,8.63%
2211,28-12-2021,212.35,219.65,220.45,211.55,2.87M,-2.75%
2212,29-12-2021,211.35,213.00,216.70,210.00,2.71M,-0.47%
2213,30-12-2021,208.50,211.45,211.50,207.90,977.48K,-1.35%


In [7]:
stk_data.rename(columns={"Price": "Close"}, inplace=True)

In [8]:
stk_data["Date"] = pd.to_datetime(stk_data["Date"])

stk_data = stk_data[
    (stk_data["Date"] >= "2020-07-01") &
    (stk_data["Date"] <= "2021-12-31")
]

C:\Users\Inst_\AppData\Local\Temp\ipykernel_3044\3978401976.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  stk_data["Date"] = pd.to_datetime(stk_data["Date"])


In [9]:
stk_data = stk_data.sort_values("Date")
stk_data = stk_data.set_index("Date")

In [10]:
stk_data

,Close,Open,High,Low,Vol.,Change %
Date,,,,,,
2020-07-01,81.95,81.50,82.70,81.05,318.89K,0.00%
2020-07-02,81.90,82.25,82.70,81.55,267.38K,-0.06%
2020-07-03,84.35,82.05,85.45,82.05,921.18K,2.99%
2020-07-06,86.60,85.80,87.85,85.05,1.66M,2.67%
2020-07-07,85.65,86.70,87.00,85.05,476.14K,-1.10%
...,...,...,...,...,...,...
2021-12-27,218.35,200.00,222.00,196.00,5.89M,8.63%
2021-12-28,212.35,219.65,220.45,211.55,2.87M,-2.75%
2021-12-29,211.35,213.00,216.70,210.00,2.71M,-0.47%


In [11]:
# As we have values with K and M in Volume(i.e string) we need to convert them into numbers
# writing a function for the same
def convert_volume(x):
    if isinstance(x,str):
        x=x.strip()
        if x.endswith('K'):
            return float(x[:-1]) * 1000
        elif x.endswith('M'):
            return float(x[:-1]) * 100000

    return float(x)

In [12]:
# using the above function to convert the volume values in the data
stk_data['Vol.']=stk_data['Vol.'].apply(convert_volume)

In [13]:
stk_data['Vol.']

Date
2020-07-01    318890.0
2020-07-02    267380.0
2020-07-03    921180.0
2020-07-06    166000.0
2020-07-07    476140.0
                ...   
2021-12-27    589000.0
2021-12-28    287000.0
2021-12-29    271000.0
2021-12-30    977480.0
2021-12-31    305000.0
Name: Vol., Length: 376, dtype: float64

In [14]:
# preprocessing
from sklearn.preprocessing import MinMaxScaler
Ms=MinMaxScaler()
cols=["Open","High","Low","Close","Vol."]
data1=pd.DataFrame(
    Ms.fit_transform(stk_data[cols]),
    columns=cols,
    index=stk_data.index
)
print("Len:",data1.shape)

Len: (376, 5)


In [15]:
# Variables to create VAR
listt=['Close','Open','High','Low']

In [16]:
# Setting exogenous variable as volume
exog=["Vol."]

In [32]:
def varmax_model(dataset, listt):

    test_obs = 28

    # Endogenous variables
    datasetTwo = dataset[listt]

    # Exogenous variable
    exog = dataset[["Vol."]]

    # Train and test
    train = datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]

    exog_train = exog[:-test_obs]
    exog_test = exog[-test_obs:]

    # Check shapes and types
    print("Train:", train.shape)
    print("Test:", test.shape)
    print("Exog train:", exog_train.shape)
    print("Exog test:", exog_test.shape)

    model = VARMAX(
        train,
        exog=exog_train,
        order=(1, 1)
    )

    result = model.fit(disp=False)

    pred = result.forecast(
        steps=28,
        exog=exog_test
    )

    print("Prediction:")
    print(pred)

    return result, pred

In [33]:
listt = ["Open", "High", "Low", "Close"]

result, pred = varmax_model(data1, listt)

Train: (348, 4)
Test: (28, 4)
Exog train: (348, 1)
Exog test: (28, 1)


C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Prediction:
           Open       High         Low       Close
348   -1.290184   0.797223    0.573378    2.457178
349   -5.673213   2.616254    7.665770    8.634447
350   -8.774574   3.600673   11.121773   12.206066
351  -13.141172   5.020264   16.682855   17.545967
352  -17.464443   6.401914   22.150728   22.689535
353  -21.665153   7.827996   27.521076   27.826926
354  -26.327038   9.311995   33.336446   33.391119
355  -31.073892  10.860312   39.384052   39.098866
356  -35.920315  12.394891   45.485004   44.829913
357  -40.779759  13.894090   51.628691   50.485300
358  -45.256558  15.311705   57.346217   55.722816
359  -49.436088  16.718572   62.683946   60.778668
360  -54.083825  18.152446   68.464112   66.228516
361  -58.513472  19.619506   74.166043   71.565367
362  -63.062359  21.067252   79.866836   76.956204
363  -67.694690  22.544477   85.743251   82.450996
364  -72.225431  24.003023   91.492270   87.833371
365  -76.780726  25.460516   97.250895   93.237038
366  -81.409772  26

C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially t